In [1]:
from collections import OrderedDict
from typing import List, Tuple, Dict, Optional
import torch.nn as nn
import torch.nn.functional as F
from torchsummary import summary
import flwr as fl
from flwr.common import NDArrays, Scalar
import torch
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from torchattacks import PGD, CW, FGSM, BIM
from torchmetrics import Precision, Recall, F1Score

In [2]:
DEVICE = torch.device("cuda")  # Try "cuda" to train on GPU
print(
    f"Training on {DEVICE} using PyTorch {torch.__version__} and Flower {fl.__version__}"
)
torch.cuda.is_available()

Training on cuda using PyTorch 2.0.0+cu118 and Flower 1.7.0


True

In [3]:
train_data = np.load("X_train.npy")
train_data = train_data.reshape((-1, 1, 180))
train_label = np.load("y_train.npy")
test_data = np.load("X_test.npy")
test_data = test_data.reshape((-1, 1, 180))
test_label = np.load("y_test.npy")

In [4]:
NUM_CLIENTS = 10

In [5]:
train_data_splits = np.array_split(train_data, NUM_CLIENTS)
train_label_splits = np.array_split(train_label, NUM_CLIENTS)

trainloaders = []
batch_size = 32

for i in range(NUM_CLIENTS):
    train_image_tensor = torch.tensor(train_data_splits[i], dtype=torch.float32)
    train_label_tensor = torch.tensor(train_label_splits[i], dtype=torch.long)
    train_dataset = TensorDataset(train_image_tensor, train_label_tensor)
    trainloader = DataLoader(train_dataset, batch_size=batch_size)

    trainloaders.append(trainloader)

In [6]:
test_data = torch.tensor(test_data, dtype=torch.float32)
test_label = torch.tensor(test_label, dtype=torch.long)
testset = TensorDataset(test_data, test_label)
testloader = DataLoader(testset, batch_size=batch_size, shuffle=False)

In [7]:
class CNN(nn.Module):
    def __init__(self, num_classes):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv1d(1, 16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool1d(2)
        self.conv3 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool1d(2)
        self.conv4 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.pool4 = nn.MaxPool1d(2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(1408, 256)
        self.fc2 = nn.Linear(256, 64)
        self.fc3 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        x = F.relu(self.conv3(x))
        x = self.pool3(x)
        x = F.relu(self.conv4(x))
        x = self.pool4(x)
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [8]:
summary(CNN(5).to(DEVICE), (1,180))

Layer (type:depth-idx)                   Output Shape              Param #
├─Conv1d: 1-1                            [-1, 16, 180]             64
├─MaxPool1d: 1-2                         [-1, 16, 90]              --
├─Conv1d: 1-3                            [-1, 32, 90]              1,568
├─MaxPool1d: 1-4                         [-1, 32, 45]              --
├─Conv1d: 1-5                            [-1, 64, 45]              6,208
├─MaxPool1d: 1-6                         [-1, 64, 22]              --
├─Conv1d: 1-7                            [-1, 128, 22]             24,704
├─MaxPool1d: 1-8                         [-1, 128, 11]             --
├─Flatten: 1-9                           [-1, 1408]                --
├─Linear: 1-10                           [-1, 256]                 360,704
├─Linear: 1-11                           [-1, 64]                  16,448
├─Linear: 1-12                           [-1, 5]                   325
Total params: 410,021
Trainable params: 410,021
Non-trainable par

Layer (type:depth-idx)                   Output Shape              Param #
├─Conv1d: 1-1                            [-1, 16, 180]             64
├─MaxPool1d: 1-2                         [-1, 16, 90]              --
├─Conv1d: 1-3                            [-1, 32, 90]              1,568
├─MaxPool1d: 1-4                         [-1, 32, 45]              --
├─Conv1d: 1-5                            [-1, 64, 45]              6,208
├─MaxPool1d: 1-6                         [-1, 64, 22]              --
├─Conv1d: 1-7                            [-1, 128, 22]             24,704
├─MaxPool1d: 1-8                         [-1, 128, 11]             --
├─Flatten: 1-9                           [-1, 1408]                --
├─Linear: 1-10                           [-1, 256]                 360,704
├─Linear: 1-11                           [-1, 64]                  16,448
├─Linear: 1-12                           [-1, 5]                   325
Total params: 410,021
Trainable params: 410,021
Non-trainable par

In [9]:
precision = Precision(num_classes=5, average='macro', task='multiclass').to(DEVICE)
recall = Recall(num_classes=5, average='macro', task='multiclass').to(DEVICE)
f1 = F1Score(num_classes=5, average='macro', task='multiclass').to(DEVICE)

In [10]:
def train(net, trainloader, epochs: int, verbose=False):
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters())
    net.train()
    for epoch in range(epochs):
        correct, total, epoch_loss = 0, 0, 0.0
        for images, labels in trainloader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            atk = PGD(net, eps=8/255, alpha=1/255, steps=20, random_start=True)
            adv_images = atk(images, labels)
            labels = torch.cat([labels, labels],0)
            images = torch.cat([adv_images, images], 0)
            outputs = net(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            epoch_loss += loss
            total += labels.size(0)
            correct += (torch.max(outputs.data, 1)[1] == labels).sum().item()
        epoch_loss /= len(trainloader)
        epoch_acc = correct / total
        if verbose:
            print(f"Epoch {epoch+1}: train loss {epoch_loss}, accuracy {epoch_acc}")


def test(net, testloader):
    criterion = torch.nn.CrossEntropyLoss()
    correct, total, loss = 0, 0, 0.0
    f1_value = 0
    recall_value = 0
    precision_value = 0
    net.eval()
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = net(images)
            loss += criterion(outputs, labels).item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            f1_value += f1(outputs, labels).item()
            recall_value += recall(outputs, labels).item()
            precision_value += precision(outputs, labels).item()
    loss /= len(testloader)
    f1_value /= len(testloader)
    recall_value /= len(testloader)
    precision_value /= len(testloader)
    accuracy = correct / total
    return loss, accuracy, f1_value, recall_value, precision_value


def eval_robust_pgd(net, testloader):
    criterion = torch.nn.CrossEntropyLoss()
    correct, total, loss = 0, 0, 0.0
    f1_value = 0
    recall_value = 0
    precision_value = 0
    net.eval()
    with torch.enable_grad():
        for images, labels in testloader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            atk = PGD(net, eps=8/255, alpha=1/255, steps=20, random_start=True)
            adv_images = atk(images, labels)
            outputs = net(adv_images)
            loss += criterion(outputs, labels).item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            f1_value += f1(outputs, labels).item()
            recall_value += recall(outputs, labels).item()
            precision_value += precision(outputs, labels).item()
    loss /= len(testloader)
    f1_value /= len(testloader)
    recall_value /= len(testloader)
    precision_value /= len(testloader)
    accuracy = correct / total
    return loss, accuracy, f1_value, recall_value, precision_value


def eval_robust_fgsm(net, testloader):
    criterion = torch.nn.CrossEntropyLoss()
    correct, total, loss = 0, 0, 0.0
    f1_value = 0
    recall_value = 0
    precision_value = 0
    net.eval()
    with torch.enable_grad():
        for images, labels in testloader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            atk = FGSM(net, eps=8/255)
            adv_images = atk(images, labels)
            outputs = net(adv_images)
            loss += criterion(outputs, labels).item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            f1_value += f1(outputs, labels).item()
            recall_value += recall(outputs, labels).item()
            precision_value += precision(outputs, labels).item()
    loss /= len(testloader)
    f1_value /= len(testloader)
    recall_value /= len(testloader)
    precision_value /= len(testloader)
    accuracy = correct / total
    return loss, accuracy, f1_value, recall_value, precision_value

def eval_robust_cw(net, testloader):
    criterion = torch.nn.CrossEntropyLoss()
    correct, total, loss = 0, 0, 0.0
    f1_value = 0
    recall_value = 0
    precision_value = 0
    net.eval()
    with torch.enable_grad():
        for images, labels in testloader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            atk = CW(net, c=1, kappa=0, steps=50, lr=0.01)
            adv_images = atk(images, labels)
            outputs = net(adv_images)
            loss += criterion(outputs, labels).item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            f1_value += f1(outputs, labels).item()
            recall_value += recall(outputs, labels).item()
            precision_value += precision(outputs, labels).item()
    loss /= len(testloader)
    f1_value /= len(testloader)
    recall_value /= len(testloader)
    precision_value /= len(testloader)
    accuracy = correct / total
    return loss, accuracy, f1_value, recall_value, precision_value

def eval_robust_bim(net, testloader):
    criterion = torch.nn.CrossEntropyLoss()
    correct, total, loss = 0, 0, 0.0
    f1_value = 0
    recall_value = 0
    precision_value = 0
    net.eval()
    with torch.enable_grad():
        for images, labels in testloader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            atk = BIM(net, eps=8/255, alpha=2/255, steps=10)
            adv_images = atk(images, labels)
            outputs = net(adv_images)
            loss += criterion(outputs, labels).item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            f1_value += f1(outputs, labels).item()
            recall_value += recall(outputs, labels).item()
            precision_value += precision(outputs, labels).item()
    loss /= len(testloader)
    f1_value /= len(testloader)
    recall_value /= len(testloader)
    precision_value /= len(testloader)
    accuracy = correct / total
    return loss, accuracy, f1_value, recall_value, precision_value

In [11]:
def get_parameters(net) -> List[np.ndarray]:
    return [val.cpu().numpy() for _, val in net.state_dict().items()]


def set_parameters(net, parameters: List[np.ndarray]):
    params_dict = zip(net.state_dict().keys(), parameters)
    state_dict = OrderedDict({k: torch.Tensor(v) for k, v in params_dict})
    net.load_state_dict(state_dict, strict=True)

In [12]:
class FlowerClient(fl.client.NumPyClient):
    def __init__(self, net, trainloader):
        self.net = net
        self.trainloader = trainloader

    def get_parameters(self, config):
        return get_parameters(self.net)

    def fit(self, parameters, config):
        set_parameters(self.net, parameters)
        train(self.net, self.trainloader, epochs=5)
        return get_parameters(self.net), len(self.trainloader), {}

In [13]:
def client_fn(cid: str) -> FlowerClient:

    net = CNN(5).to(DEVICE)

    trainloader = trainloaders[int(cid)]

    return FlowerClient(net, trainloader).to_client()

In [14]:
def evaluate(
    server_round: int,
    parameters: NDArrays,
    config: Dict[str, Scalar],
) -> Optional[Tuple[float, Dict[str, Scalar]]]:
    net = CNN(5).to(DEVICE)
    set_parameters(net, parameters)  # Update model with the latest parameters
    loss, clean, f1_value, recall_value, precision_value = test(net, testloader)
    loss_pgd, robust_pgd, f1_pgd, recall_pgd, precision_pgd = eval_robust_pgd(net, testloader)
    loss_fgsm, robust_fgsm, f1_fgsm, recall_fgsm, precision_fgsm = eval_robust_fgsm(net, testloader)   
    loss_cw, robust_cw, f1_cw, recall_cw, precision_cw = eval_robust_cw(net, testloader)      
    loss_bim, robust_bim, f1_bim, recall_bim, precision_bim = eval_robust_bim(net, testloader)
    return loss, {"clean_acc": clean, 
                  "clean_f1": f1_value,
                  "clean_recall": recall_value,
                  "clean_precision": precision_value,
                  "pgd_acc": robust_pgd, 
                  "pgd_loss": loss_pgd,
                  "pgd_f1": f1_pgd,
                  "pgd_recall": recall_pgd,
                  "pgd_precision": precision_pgd,
                  "fgsm_acc": robust_fgsm, 
                  "fgsm_loss": loss_fgsm,
                  "fgsm_f1": f1_fgsm,
                  "fgsm_recall": recall_fgsm,
                  "fgsm_precision": precision_fgsm,
                  "cw_acc": robust_cw, 
                  "cw_loss": loss_cw,
                  "cw_f1": f1_cw,
                  "cw_recall": recall_cw,
                  "cw_precision": precision_cw,
                  "bim_acc": robust_bim, 
                  "bim_loss": loss_bim,
                  "bim_f1": f1_bim,
                  "bim_recall": recall_bim,
                  "bim_precision": precision_bim}

In [15]:
strategy = fl.server.strategy.FedAvg(
    fraction_fit=0.5,
    fraction_evaluate=0,
    min_fit_clients=5,
    min_evaluate_clients=0,
    min_available_clients=10,
    evaluate_fn=evaluate
)

client_resources = {"num_cpus": 6.0, "num_gpus": 0.5}


history = fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=NUM_CLIENTS,
    config=fl.server.ServerConfig(num_rounds=10),
    strategy=strategy,
    client_resources=client_resources,
)

INFO flwr 2024-10-14 21:02:02,112 | app.py:178 | Starting Flower simulation, config: ServerConfig(num_rounds=10, round_timeout=None)
2024-10-14 21:02:06,180	INFO worker.py:1621 -- Started a local Ray instance.
INFO flwr 2024-10-14 21:02:07,997 | app.py:213 | Flower VCE: Ray initialized with resources: {'object_store_memory': 1497434112.0, 'memory': 2994868224.0, 'node:127.0.0.1': 1.0, 'CPU': 12.0, 'GPU': 1.0, 'node:__internal_head__': 1.0}
INFO flwr 2024-10-14 21:02:07,998 | app.py:219 | Optimize your simulation with Flower VCE: https://flower.dev/docs/framework/how-to-run-simulations.html
INFO flwr 2024-10-14 21:02:07,999 | app.py:242 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 6.0, 'num_gpus': 0.5}
INFO flwr 2024-10-14 21:02:08,007 | app.py:288 | Flower VCE: Creating VirtualClientEngineActorPool with 2 actors
INFO flwr 2024-10-14 21:02:08,008 | server.py:89 | Initializing global parameters
INFO flwr 2024-10-14 21:02:08,009 | server.py:276 | Requesting initial parame

INFO flwr 2024-10-14 21:29:47,513 | server.py:171 | evaluate_round 5: no clients selected, cancel
DEBUG flwr 2024-10-14 21:29:47,514 | server.py:222 | fit_round 6: strategy sampled 5 clients (out of 10)
DEBUG flwr 2024-10-14 21:32:11,881 | server.py:236 | fit_round 6 received 5 results and 0 failures
INFO flwr 2024-10-14 21:35:05,315 | server.py:125 | fit progress: (6, 0.09787179028656143, {'clean_acc': 0.9739654699917786, 'clean_f1': 0.8802323161685554, 'clean_recall': 0.8775443112763175, 'clean_precision': 0.89575480414133, 'pgd_acc': 0.9470631223166164, 'pgd_loss': 0.198312645599953, 'pgd_f1': 0.7926034198190174, 'pgd_recall': 0.7951156839402053, 'pgd_precision': 0.8140206640654237, 'fgsm_acc': 0.9514478852653695, 'fgsm_loss': 0.18151714853045062, 'fgsm_f1': 0.8062883193040417, 'fgsm_recall': 0.8064389860107951, 'fgsm_precision': 0.8286515158458347, 'cw_acc': 0.4015255321092537, 'cw_loss': 0.7848671273158414, 'cw_f1': 0.19964275753846134, 'cw_recall': 0.2227099006900387, 'cw_precisi

In [16]:
history

History (loss, centralized):
	round 0: 1.6228875299439813
	round 1: 0.4429889982634217
	round 2: 0.1645734233849675
	round 3: 0.12263255401031814
	round 4: 0.1116194088039172
	round 5: 0.10223924235957436
	round 6: 0.09787179028656143
	round 7: 0.09673954658686136
	round 8: 0.09722417690111392
	round 9: 0.09743962788301772
	round 10: 0.09276287082045673
History (metrics, centralized):
{'clean_acc': [(0, 0.007033890563624737), (1, 0.8951767607563716), (2, 0.9546451082488353), (3, 0.965515666392619), (4, 0.9699004293413721), (5, 0.9723668585000457), (6, 0.9739654699917786), (7, 0.9744222161322736), (8, 0.9743308669041747), (9, 0.9751073353430163), (10, 0.9760665022380561)], 'clean_f1': [(0, 0.003177941564715257), (1, 0.5333706943005541), (2, 0.8098592358349014), (3, 0.8486591506178361), (4, 0.8638974322454773), (5, 0.8739654079841002), (6, 0.8802323161685554), (7, 0.8815720381092851), (8, 0.8835378587245941), (9, 0.8867348913728755), (10, 0.8906688445240912)], 'clean_recall': [(0, 0.0467